## Executive Summary

**Purpose:** One call from the requirements (ratio, torque, speed, life, envelope, input shaft) to a complete, checked single-stage planetary design and the files needed to make it.

**What it does:** `design_planetary` searches every tooth-count set near the target ratio, for each allowed number of planets and each standard module that fits between the sun bore and the envelope, and finds the narrowest face width at which every check passes: mesh geometry, ISO 6336 pitting and bending, planet bearing life, pin, carrier, key, ring fixing and axial length. It returns the passing designs ranked. `PlanetaryDesign.write_package` writes, for the chosen design:

| File | Content |
|---|---|
| `report.md` | requirements, chosen design, every check with its value and limit, assumptions |
| `gear_data_sun.csv`, `gear_data_planet.csv`, `gear_data_ring.csv` | ISO 1340 / DIN 3966-1 gear data tables for the drawings |
| `parts.csv` | parts list with quantities, key dimensions and fits |
| `component_dimensions.csv` | dimensions and limits of pin, bore, carrier, key, ring fixing |
| `layout.svg` | front view and axial section of the stage |
| `sun.dxf`, `planet.dxf`, `ring.dxf` | tooth outlines at mean thickness, for prototypes and drawings |

**Inputs ND supplies:** outer envelope (diameter, optionally length) and the shaft diameter that carries the sun. The rest have defaults that are stated in the report and can all be overridden.

## How It Fits Into the Gear Design Workflow

**Upstream:** all other `gears` notebooks: `involute` → `layout` → `rating` → `components` → `manufacturing`.

**Downstream:** the drawings. A designer copies the data tables onto the gear drawings, the fits onto the part drawings, and uses the layout for the assembly drawing.

In [ ]:
#| default_exp gears.design

In [ ]:
#| export
from __future__ import annotations

import csv
import math
import os
from dataclasses import dataclass, field, asdict, replace

import numpy as np

from emachines.gears.involute import GearGeometry, standard_modules
from emachines.gears.layout import PlanetarySet, tooth_count_candidates, module_limits
from emachines.gears.rating import MATERIALS, GearMaterial, RatingInputs, PlanetaryRating, rate_planetary
from emachines.gears.components import (ComponentInputs, PlanetaryComponents, ParallelKey, rim_thicknesses,
                                        size_components, _hub_depth, BOLTS_88)
from emachines.gears.manufacturing import (backlash_allowances, gear_data_sheet, gear_outline, write_dxf,
                                           iso286_limits, fmt_fit, it_grade, DataRow)

## Requirements

In [ ]:
#| export
@dataclass(frozen=True)
class PlanetaryRequirements:
    """
    What the stage must do and where it must fit. Lengths mm, torque N·m, speed rpm.

    ratio, torque_out, n_in, life_h : target ratio, output torque for rating, input speed, required life [h]
    envelope_d, envelope_length     : outer diameter and axial length available (length optional)
    shaft_d                         : input shaft diameter in the sun bore
    hub_joint                       : 'key', 'spline' or 'integral'
    ratio_tol                       : allowed relative ratio error
    n_planets                       : planet counts to try
    K_A                             : application factor
    materials                       : {'sun'|'planet'|'ring': key of gears.rating.MATERIALS}
    grade, ring_grade               : ISO 1328-1 flank tolerance class of sun/planets and of the ring
    flank_finish                    : 'ground' or 'hobbed' for sun and planets
    S_Hmin, S_Fmin                  : required safety factors
    floating_sun                    : the sun is radially free (load sharing)
    housing_wall                    : housing wall outside the ring (0: ring is the outer wall)
    series_ii                       : also allow ISO 54 series II modules
    zs_range                        : sun tooth counts to search
    """
    ratio: float
    torque_out: float
    n_in: float
    envelope_d: float
    shaft_d: float
    envelope_length: float | None = None
    life_h: float = 10_000.0
    hub_joint: str = "key"
    ratio_tol: float = 0.03
    n_planets: tuple[int, ...] = (3, 4, 5)
    K_A: float = 1.25
    materials: dict = field(default_factory=lambda: {"sun": "case_carburized", "planet": "case_carburized",
                                                     "ring": "qt_nitrided"})
    grade: int = 6
    ring_grade: int = 7
    flank_finish: str = "ground"
    S_Hmin: float = 1.0
    S_Fmin: float = 1.4
    floating_sun: bool = False
    housing_wall: float = 0.0
    series_ii: bool = True
    zs_range: tuple[int, int] = (12, 40)

    def rating_inputs(self) -> RatingInputs:
        return RatingInputs(torque_out=self.torque_out, n_in=self.n_in, life_h=self.life_h, K_A=self.K_A,
                            grade=self.grade, floating_sun=self.floating_sun, S_Hmin=self.S_Hmin,
                            S_Fmin=self.S_Fmin, flank_finish=self.flank_finish)

    def component_inputs(self) -> ComponentInputs:
        return ComponentInputs(envelope_d=self.envelope_d, shaft_d=self.shaft_d,
                               envelope_length=self.envelope_length, hub_joint=self.hub_joint,
                               housing_wall=self.housing_wall)

    def material_set(self) -> dict[str, GearMaterial]:
        return {k: MATERIALS[v] for k, v in self.materials.items()}

## The Search: Theory

For every number of planets $N$ and every tooth-count set within the ratio tolerance (`tooth_count_candidates`):

1. **Module window.** `module_limits` with the hard ISO 6336-3 rim limits (sun rim ≥ 0.5 tooth depths over the key groove, ring rim ≥ 1.75 m). Thinner-than-ideal rims are allowed and charged through $Y_B$. Every standard module in the window is tried.
2. **Geometry.** `PlanetarySet.build`; sets that fail a mesh, neighbour or tip-thickness check are dropped.
3. **Face width.** Starting from $b = 6m$ (but at least 5 mm; narrower teeth are hard to hold square and to inspect) and growing in 1 mm steps up to $\min(1.2\,d_s,\ 0.8\,d_p\cdot 1.5,\ L_{env} - L_{fixed})$, each width is rated (two passes, so that the planet rim matches the bearing chosen) and its components sized. The first width at which everything passes is kept. Gears are only as wide as they need to be; extra width costs length and makes the load less even across the face.
4. **Ranking.** Passing designs are ordered by ratio error (in 0.5 % steps), then by axial length, then by the lowest margin over the required safety factors (larger first).

The search typically takes a few seconds and evaluates a few hundred designs.

In [ ]:
#| export
@dataclass
class PlanetaryDesign:
    "A complete checked design: requirements, gear set, rating, components and backlash allowances."
    req: PlanetaryRequirements
    gears: PlanetarySet
    rating: PlanetaryRating
    components: PlanetaryComponents
    allowances: dict

    @property
    def margin(self) -> float:
        "Smallest ratio of achieved to required safety factor."
        return min(self.rating.S_H / self.req.S_Hmin, self.rating.S_F / self.req.S_Fmin)

    @property
    def ratio_error(self) -> float:
        return (self.gears.ratio - self.req.ratio) / self.req.ratio

    @property
    def length(self) -> float:
        return self.components.axial["stack"]

    def label(self) -> str:
        g = self.gears
        return (f"{g.sun.z}/{g.planet.z}/{g.ring.z} ×{g.n_planets}, m {g.sun.m:g}, b {g.sun.b:g} mm: "
                f"i = {g.ratio:.4f} ({self.ratio_error * 100:+.2f} %), S_H {self.rating.S_H:.2f}, "
                f"S_F {self.rating.S_F:.2f}, length {self.length:.1f} mm, limited by {self.rating.limiting}")


def evaluate(req: PlanetaryRequirements, zs: int, zp: int, zr: int, m: float, n_planets: int,
             b: float) -> PlanetaryDesign | None:
    "Build, rate and size one design; None if the geometry itself is not valid."
    try:
        ps = PlanetarySet.build(zs, zp, zr, m=m, n_planets=n_planets, b=b)
    except ValueError:
        return None
    if not ps.ok:
        return None
    ci = req.component_inputs()
    inp = req.rating_inputs()
    mats = req.material_set()
    try:
        r0 = rate_planetary(ps, inp, mats, rim_thicknesses(ps, ci))
        c0 = size_components(ps, r0, ci, req.life_h, req.n_in, req.K_A)
        rims = rim_thicknesses(ps, ci, planet_bore=c0.bearing["Ew"])
        r = rate_planetary(ps, inp, mats, rims)
        c = size_components(ps, r, ci, req.life_h, req.n_in, req.K_A)
    except ValueError:
        return None
    bl = backlash_allowances(ps, req.grade)
    return PlanetaryDesign(req, ps, r, c, bl)


def design_planetary(req: PlanetaryRequirements, max_results: int = 10) -> list[PlanetaryDesign]:
    "Search tooth counts, planets, modules and face widths; return passing designs, best first (see theory)."
    ci = req.component_inputs()
    t2 = _hub_depth(ci)
    found: list[PlanetaryDesign] = []
    for N in req.n_planets:
        for c in tooth_count_candidates(req.ratio, N, req.ratio_tol, req.zs_range):
            m_lo, m_hi = module_limits(req.envelope_d, req.shaft_d if req.hub_joint != "integral" else 0.0,
                                       c.zs, c.zr, t2=t2, housing_wall=req.housing_wall,
                                       ring_rim=1.75, sun_rim=0.5)
            for m in standard_modules(m_lo, m_hi, series_ii=req.series_ii):
                b_lo = max(5.0, math.ceil(6 * m))
                b_hi = 1.2 * c.zs * m
                if req.envelope_length is not None:
                    b_hi = min(b_hi, req.envelope_length - 2 * 4.0 - 2 * 1.0 - 2 * 0.5)
                b = b_lo
                while b <= b_hi + 1e-9:
                    d = evaluate(req, c.zs, c.zp, c.zr, m, N, b)
                    if d is None:
                        break
                    if d.rating.ok and d.components.ok:
                        found.append(d)
                        break
                    b += 1.0
    found.sort(key=lambda d: (round(abs(d.ratio_error) / 0.005), round(d.length), -d.margin))
    return found[:max_results]

## Drawing: Front View and Axial Section

`layout_svg` draws the stage to scale: on the left the front view with true involute teeth, carrier plate outline, pin holes, the input shaft with its key, and the ring outer diameter; on the right the axial half-section with the carrier plates, gears, pins, bearings and thrust washers. Main dimensions are written beside the views.

In [ ]:
#| export
def _poly(P: np.ndarray, sx: float, ox: float, oy: float, style: str) -> str:
    pts = " ".join(f"{ox + x * sx:.2f},{oy - y * sx:.2f}" for x, y in P)
    return f'<polygon points="{pts}" {style}/>'


def layout_svg(d: PlanetaryDesign, width_px: int = 1100) -> str:
    "SVG text of the front view and axial section of a design (mm scale, see above)."
    g, comp = d.gears, d.components
    R = d.req.envelope_d / 2 - d.req.housing_wall
    L = comp.axial["stack"]
    view_w = 2 * R + 30 + L + 25 + 150
    sx = width_px / view_w
    H = (2 * R + 40) * sx
    ox, oy = (R + 10) * sx, (R + 20) * sx
    thin = 'fill="none" stroke="#222" stroke-width="0.8"'
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width_px}" height="{H + 60:.0f}" font-family="sans-serif" font-size="11">',
           '<rect width="100%" height="100%" fill="white"/>']
    ds = {k: (v.A_sne + v.A_sni) / 2 for k, v in d.allowances["allowances"].items()}
    # ring outer circle and teeth
    out.append(f'<circle cx="{ox:.2f}" cy="{oy:.2f}" r="{R * sx:.2f}" fill="#e8eef5" stroke="#222" stroke-width="1"/>')
    rho = np.pi * g.planet.z / g.ring.z                          # ring phase that meshes with the planets
    Rr = np.array([[np.cos(rho), np.sin(rho)], [-np.sin(rho), np.cos(rho)]])
    out.append(_poly(gear_outline(g.ring, ds["ring"]) @ Rr, sx, ox, oy, 'fill="white" stroke="#222" stroke-width="0.8"'))
    # carrier plate (behind), dashed
    out.append(f'<circle cx="{ox:.2f}" cy="{oy:.2f}" r="{comp.carrier["plate_d"] / 2 * sx:.2f}" fill="none" stroke="#2a6" stroke-dasharray="5,3"/>')
    # planets
    for k in range(g.n_planets):
        th = 2 * np.pi * k / g.n_planets
        psi = th * (1 + g.sun.z / g.planet.z) + np.pi - np.pi / g.planet.z   # planet phase that meshes with the sun
        c, s = np.cos(psi), np.sin(psi)
        P = gear_outline(g.planet, ds["planet"]) @ np.array([[c, s], [-s, c]])
        cx, cy = g.a_w * np.cos(th), g.a_w * np.sin(th)
        out.append(_poly(P + [cx, cy], sx, ox, oy, 'fill="#f6e7c8" stroke="#222" stroke-width="0.8"'))
        out.append(f'<circle cx="{ox + cx * sx:.2f}" cy="{oy - cy * sx:.2f}" r="{comp.bearing["Ew"] / 2 * sx:.2f}" fill="white" stroke="#222" stroke-width="0.6"/>')
        out.append(f'<circle cx="{ox + cx * sx:.2f}" cy="{oy - cy * sx:.2f}" r="{comp.pin["d"] / 2 * sx:.2f}" fill="#bbb" stroke="#222" stroke-width="0.6"/>')
    # sun, shaft, key
    out.append(_poly(gear_outline(g.sun, ds["sun"]), sx, ox, oy, 'fill="#d8e8d0" stroke="#222" stroke-width="0.8"'))
    out.append(f'<circle cx="{ox:.2f}" cy="{oy:.2f}" r="{d.req.shaft_d / 2 * sx:.2f}" fill="#bbb" stroke="#222" stroke-width="0.6"/>')
    if d.req.hub_joint == "key":
        key = ParallelKey.for_shaft(d.req.shaft_d)
        y0 = d.req.shaft_d / 2 - key.t1
        out.append(f'<rect x="{ox - key.b / 2 * sx:.2f}" y="{oy - (y0 + key.h) * sx:.2f}" width="{key.b * sx:.2f}" height="{key.h * sx:.2f}" fill="#888" stroke="#222" stroke-width="0.5"/>')
    out.append(f'<line x1="{ox - (R + 5) * sx:.1f}" y1="{oy:.1f}" x2="{ox + (R + 5) * sx:.1f}" y2="{oy:.1f}" stroke="#a33" stroke-width="0.4" stroke-dasharray="8,2,2,2"/>')
    out.append(f'<line x1="{ox:.1f}" y1="{oy - (R + 5) * sx:.1f}" x2="{ox:.1f}" y2="{oy + (R + 5) * sx:.1f}" stroke="#a33" stroke-width="0.4" stroke-dasharray="8,2,2,2"/>')
    out.append(f'<text x="{ox:.1f}" y="{H + 15:.0f}" text-anchor="middle">Front view (carrier plate dashed)</text>')
    # axial half section, axis horizontal at the bottom of the section
    ax0 = ox + (R + 30) * sx
    base = oy                       # axis line
    t_c, b, w = comp.carrier["plate_t"], g.sun.b, 1.0
    cl = 0.5
    z = ax0
    def rect(x, r0, r1, length, fill):
        return (f'<rect x="{x:.2f}" y="{base - r1 * sx:.2f}" width="{length * sx:.2f}" height="{(r1 - r0) * sx:.2f}" '
                f'fill="{fill}" stroke="#222" stroke-width="0.6"/>')
    x_pl1 = z + cl * sx
    x_g = x_pl1 + (t_c + w) * sx
    x_pl2 = x_g + (b + w) * sx
    out.append(rect(z, g.ring.df / 2, R, L, "#e8eef5"))                                       # ring body
    out.append(rect(x_g, g.ring.da / 2, g.ring.df / 2, b, "#cfd9e6"))                          # ring teeth
    out.append(rect(x_pl1, g.sun.da / 2 + 1, comp.carrier["plate_d"] / 2, t_c, "#cfe8d6"))     # plates
    out.append(rect(x_pl2, g.sun.da / 2 + 1, comp.carrier["plate_d"] / 2, t_c, "#cfe8d6"))
    out.append(rect(x_g, g.a_w - g.planet.da / 2, g.a_w + g.planet.da / 2, b, "#f6e7c8"))     # planet
    out.append(rect(x_pl1, g.a_w - comp.pin["d"] / 2, g.a_w + comp.pin["d"] / 2, 2 * t_c + b + 2 * w, "#bbb"))   # pin
    out.append(rect(x_g + 0.5 * sx, g.a_w + comp.pin["d"] / 2, g.a_w + comp.bearing["Ew"] / 2, comp.bearing["Bc"], "#999"))  # needles
    out.append(rect(x_g, d.req.shaft_d / 2, g.sun.da / 2, b, "#d8e8d0"))                       # sun
    out.append(rect(z - 10 * sx, 0, d.req.shaft_d / 2, L + 20, "#ddd"))                        # shaft
    out.append(f'<line x1="{z - 15 * sx:.1f}" y1="{base:.1f}" x2="{z + (L + 15) * sx:.1f}" y2="{base:.1f}" stroke="#a33" stroke-width="0.4" stroke-dasharray="8,2,2,2"/>')
    out.append(f'<text x="{z + L / 2 * sx:.1f}" y="{H + 15:.0f}" text-anchor="middle">Axial half-section (input at left)</text>')
    # data block
    tx = z + (L + 25) * sx
    lines = [f"i = {g.ratio:.4f}  ({g.sun.z}/{g.planet.z}/{g.ring.z}, {g.n_planets} planets)",
             f"module {g.sun.m:g} mm, α = 20°, face {b:g} mm",
             f"a_w = {g.a_w:.4f} mm (pin circle Ø{2 * g.a_w:.3f})",
             f"ring Ø{2 * R:g} × {L:.1f} mm stack",
             f"sun on Ø{d.req.shaft_d:g} shaft, {d.components.sun_hub.get('key', d.req.hub_joint)}",
             f"planet bearing {comp.bearing['type'].split(' (')[0].replace('needle roller cage assembly ', '')}",
             f"pin Ø{comp.pin['d']:g} j5 × {comp.pin['length']:g}",
             f"carrier plates Ø{comp.carrier['plate_d']:g} × {t_c:g}",
             f"T_out {d.req.torque_out:g} N·m at {d.req.n_in / g.ratio:.0f} rpm",
             f"S_H {d.rating.S_H:.2f} / S_F {d.rating.S_F:.2f}",
             f"L10h planet bearing ≈ {comp.bearing['L10h_est']:.3g} h"]
    for i, s in enumerate(lines):
        out.append(f'<text x="{tx:.1f}" y="{30 + 16 * i}">{s}</text>')
    out.append("</svg>")
    return "\n".join(out)

## Report and Package

In [ ]:
#| export
def _fmt(v) -> str:
    if isinstance(v, float):
        return f"{v:.4g}"
    return str(v)


def component_rows(d: PlanetaryDesign) -> list[tuple[str, str, str]]:
    "(part, feature, specification) rows with numeric limits for the part drawings."
    c, g = d.components, d.gears
    Fw, Ew = c.bearing["Fw"], c.bearing["Ew"]
    rows = [("planet", "bore (bearing outer raceway)", fmt_fit(Ew, "G6") + ", 58–64 HRC, Ra ≤ 0.2 µm, cylindricity 0.003"),
            ("planet", "faces", f"run-out to bore ≤ 0.01 mm, width {g.planet.b:g} −0.05/−0.10 mm"),
            ("planet pin", "diameter", fmt_fit(Fw, "j5") + ", 58–64 HRC, Ra ≤ 0.2 µm"),
            ("planet pin", "length", f"{c.pin['length']:g} mm, chamfer 0.5 × 45° both ends"),
            ("carrier plates", "pin holes", f"{g.n_planets} × " + fmt_fit(Fw, "N7") + f" on Ø{2 * g.a_w:.3f} pin circle"),
            ("carrier plates", "pin hole position", f"position Ø{2 * it_grade(g.a_w, 7) / 2:.3f} mm to bore axis (= ±f_a on centre distance); "
                                                   f"spacing {360 / g.n_planets:g}° ±{math.degrees(it_grade(g.a_w, 7) / 2 / g.a_w) * 60:.1f}′; "
                                                   "both plates line-bored together"),
            ("carrier plates", "outside diameter", f"Ø{c.carrier['plate_d']:g} h11, thickness {c.carrier['plate_t']:g} mm"),
            ("carrier plates", "centre bore (sun clearance)", f"Ø{math.ceil(c.carrier['sun_clearance_bore']):g} mm min"),
            ("carrier", "posts", f"{g.n_planets} posts {c.carrier['post_width']:.1f} wide × {c.carrier['post_height']:.1f} radial × {c.carrier['post_length']:.1f} long"),
            ("carrier", "output", f"output shaft Ø{math.ceil(c.carrier['output_shaft_min_d']):g} mm min (torsion), bearing seats per housing design"),
            ("thrust washer", "size", c.axial["thrust_washers"]),
            ]
    hub = c.sun_hub
    if hub["joint"] == "key":
        d_sh = hub["shaft_d"]
        rows += [("sun", "bore", fmt_fit(d_sh, "H7") + " (shaft k6)"),
                 ("sun", "key groove", f"width {hub['key_b']:g} P9 {_dev(hub['key_b'], 'P9')}, depth {d_sh + hub['t2']:.1f} +0.1 over bore side"),
                 ("key", "parallel key", hub["key"] + ", C45+C"),
                 ("input shaft", "key seat", f"width {hub['key_b']:g} P9, depth t1 = {hub['t1']:g} +0.1 mm")]
    else:
        rows.append(("sun", "bore", hub.get("spline", "integral with shaft")))
    rm = c.ring_mount
    rows.append(("ring", "outside diameter", f"Ø{rm['ring_od']:g} " + ("h7" if rm["bolts"] else "s6 (press into housing H7)")))
    rows.append(("ring", "fixing", rm["method"] + (f", tightening torque {rm['tightening_torque']:g} N·m" if rm["bolts"] else "")))
    return rows


def _dev(dn: float, fit: str) -> str:
    up, lo = iso286_limits(dn, fit)
    f = lambda v: "0" if abs(v) < 5e-7 else f"{v:+.3f}"
    return f"({f(up)}/{f(lo)})"


def data_sheets(d: PlanetaryDesign) -> dict[str, list[DataRow]]:
    "Gear data tables of sun, planet and ring."
    g, al = d.gears, d.allowances["allowances"]
    mats = d.req.material_set()
    fin = d.req.flank_finish
    return {
        "sun": gear_data_sheet(g.sun, g.planet, g.a_w, al["sun"], d.req.grade, mats["sun"], fin,
                               f" ({g.n_planets} planets)", d_active=g.sun_planet.d_Nf1),
        "planet": gear_data_sheet(g.planet, g.sun, g.a_w, al["planet"], d.req.grade, mats["planet"], fin,
                                  f", and ring z = {g.ring.z}", d_active=min(g.sun_planet.d_Nf2, g.planet_ring.d_Nf1)),
        "ring": gear_data_sheet(g.ring, g.planet, g.a_w, al["ring"], d.req.ring_grade, mats["ring"], "shaped",
                                f" ({g.n_planets} planets)", d_active=g.planet_ring.d_Nf2),
    }


def parts_list(d: PlanetaryDesign) -> list[tuple[int, str, int, str, str]]:
    "(item, part, qty, description, material/standard)."
    g, c = d.gears, d.components
    mats = d.req.material_set()
    N = g.n_planets
    items = [("Sun gear", 1, f"z {g.sun.z}, m {g.sun.m:g}, b {g.sun.b:g}, bore Ø{d.req.shaft_d:g}", mats["sun"].name + ", " + mats["sun"].treatment),
             ("Planet gear", N, f"z {g.planet.z}, m {g.planet.m:g}, b {g.planet.b:g}, bore Ø{c.bearing['Ew']:g} G6", mats["planet"].name + ", " + mats["planet"].treatment),
             ("Ring gear", 1, f"z {g.ring.z}, m {g.ring.m:g}, b {g.ring.b:g}, OD Ø{c.ring_mount['ring_od']:g}", mats["ring"].name + ", " + mats["ring"].treatment),
             ("Planet bearing", N, c.bearing["type"], f"C_req {c.bearing['C_req'] / 1000:.2f} kN"),
             ("Planet pin", N, f"Ø{c.pin['d']:g} j5 × {c.pin['length']:g}", c.pin["material"]),
             ("Thrust washer", 2 * N, c.axial["thrust_washers"], ""),
             ("Carrier, output plate with posts", 1, f"Ø{c.carrier['plate_d']:g} × {c.carrier['plate_t']:g} + {N} posts", c.carrier["material"]),
             ("Carrier, free plate", 1, f"Ø{c.carrier['plate_d']:g} × {c.carrier['plate_t']:g}, screwed and doweled to the posts", c.carrier["material"]),
             ]
    if c.sun_hub["joint"] == "key":
        items.append(("Parallel key", 1, c.sun_hub["key"], "C45+C, DIN 6885-1"))
    if c.ring_mount["bolts"]:
        items.append(("Ring clamping bolt", c.ring_mount["bolts"], f"{d.components.ring_mount['method'].split(' through')[0].split('× ')[1]} socket head, ISO 4762", "8.8"))
        items.append(("Dowel pin", 2, "ISO 8734, locate ring", "hardened"))
    else:
        items.append(("Dowel pin", 2, "ISO 8734, secure pressed ring against turning", "hardened"))
    return [(i + 1, *row) for i, row in enumerate(items)]


def report_markdown(d: PlanetaryDesign) -> str:
    "The design report (see the package table above)."
    r, g, c, req = d.rating, d.gears, d.components, d.req
    sp = g.speeds(req.n_in)
    L = [f"# Single-stage planetary gear: design report", "",
         "Ring gear fixed, sun input, carrier output. Spur gears, 20° pressure angle, ISO 53 profile A. "
         "Lengths in mm, torques in N·m, stresses in MPa. Calculated with `emachines.gears`.", "",
         "## Requirements", "",
         "| Item | Value |", "|---|---|"]
    for k, v in asdict(req).items():
        L.append(f"| {k} | {v} |")
    L += ["", "## Gear set", "", "```", g.summary(), "```", "",
          f"Speeds at {req.n_in:g} rpm input: carrier {sp['carrier']:.1f} rpm, planets {sp['planet_rel']:.1f} rpm on their pins "
          f"({sp['planet_abs']:.1f} rpm absolute). Mesh frequency {g.mesh_frequency(req.n_in):.0f} Hz.", "",
          f"Torques: sun {r.T_sun:.3f}, carrier {r.T_carrier:.3f}, ring reaction {r.T_ring:.3f}. "
          f"Tangential force per planet mesh {r.F_t:.1f} N (before K_γ). Pitch line velocity {r.v:.2f} m/s.", "",
          "## Load capacity (ISO 6336)", "", "Factors: " + ", ".join(f"{k} = {v:.3f}" for k, v in r.factors.items()), "",
          "```", r.table(), "```", "",
          "## Backlash and tooth thickness", "",
          f"Required minimum normal backlash {d.allowances['j_bn_min_required']:.3f} mm; centre distance tolerance ±{d.allowances['f_a'] * 1000:.0f} µm.", "",
          "| Gear | A_sne | A_sni |", "|---|---|---|"]
    for k, a in d.allowances["allowances"].items():
        L.append(f"| {k} | {a.A_sne:+.3f} | {a.A_sni:+.3f} |")
    for mesh, (jmin, jmax) in d.allowances["backlash_n"].items():
        L.append(f"\n{mesh}: normal backlash {jmin:.3f} to {jmax:.3f} mm.")
    for name, rows in data_sheets(d).items():
        L += ["", f"## Gear data — {name}", "", "| Item | Symbol | Value |", "|---|---|---|"]
        L += [f"| {x.item} | {x.symbol} | {x.value} |" for x in rows]
    L += ["", "## Components", ""]
    L += ["| Part | Feature | Specification |", "|---|---|---|"] + [f"| {a} | {b_} | {s} |" for a, b_, s in component_rows(d)]
    L += ["", "### Planet bearing", ""] + [f"- {k}: {_fmt(v)}" for k, v in c.bearing.items()]
    L += ["", "### Planet pin", ""] + [f"- {k}: {_fmt(v)}" for k, v in c.pin.items()]
    L += ["", "### Carrier", ""] + [f"- {k}: {_fmt(v)}" for k, v in c.carrier.items()]
    L += ["", "### Sun hub", ""] + [f"- {k}: {_fmt(v)}" for k, v in c.sun_hub.items()]
    L += ["", "### Ring fixing", ""] + [f"- {k}: {_fmt(v)}" for k, v in c.ring_mount.items()]
    L += ["", "### Axial stack", ""] + [f"- {k}: {_fmt(v)}" for k, v in c.axial.items()]
    L += ["", "## Checks", ""] + [f"- {ch}" for ch in c.checks]
    L += [f"- {'✓' if r.ok else '✗'} gear rating: S_H {r.S_H:.2f} ≥ {req.S_Hmin}, S_F {r.S_F:.2f} ≥ {req.S_Fmin}"]
    L += ["", "## Lubrication", "", c.lubrication, "",
          "## Parts list", "", "| # | Part | Qty | Description | Material / standard |", "|---|---|---|---|---|"]
    L += [f"| {i} | {p} | {q} | {ds} | {mt} |" for i, p, q, ds, mt in parts_list(d)]
    L += ["", "## Assumptions to confirm before release", "",
          "- Material allowables are ISO 6336-5 MQ typical values; confirm with the steel certificate and heat treater.",
          f"- K_Hβ = {req.rating_inputs().K_Hbeta} assumes a stiff double-plate carrier and crowned or ground planets.",
          "- The planet bearing rating is an ISO 281 estimate; select the catalogue part with C_r ≥ C_req and check its speed limit.",
          "- Tooth thickness tolerances follow DIN 3967 practice; agree them with the gear shop.",
          "- The ring's root form diameter is estimated from a rack; confirm with the shaper cutter data.",
          "- Re-check the final design in a full ISO 6336 tool if any safety factor is within 10 % of its minimum."]
    return "\n".join(L) + "\n"


def write_package(d: PlanetaryDesign, folder: str) -> list[str]:
    "Write the manufacturing package (see the table above) into `folder`; returns the file paths."
    os.makedirs(folder, exist_ok=True)
    paths = []
    def p(name):
        paths.append(os.path.join(folder, name))
        return paths[-1]
    with open(p("report.md"), "w") as f:
        f.write(report_markdown(d))
    for name, rows in data_sheets(d).items():
        with open(p(f"gear_data_{name}.csv"), "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["item", "symbol", "value"])
            w.writerows([(x.item, x.symbol, x.value) for x in rows])
    with open(p("parts.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["item", "part", "qty", "description", "material_or_standard"])
        w.writerows(parts_list(d))
    with open(p("component_dimensions.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["part", "feature", "specification"])
        w.writerows(component_rows(d))
    with open(p("layout.svg"), "w") as f:
        f.write(layout_svg(d))
    al = d.allowances["allowances"]
    g = d.gears
    write_dxf(p("sun.dxf"), [gear_outline(g.sun, (al["sun"].A_sne + al["sun"].A_sni) / 2)], [(0, 0, d.req.shaft_d / 2)])
    write_dxf(p("planet.dxf"), [gear_outline(g.planet, (al["planet"].A_sne + al["planet"].A_sni) / 2)],
              [(0, 0, d.components.bearing["Ew"] / 2)])
    write_dxf(p("ring.dxf"), [gear_outline(g.ring, (al["ring"].A_sne + al["ring"].A_sni) / 2)],
              [(0, 0, d.components.ring_mount["ring_od"] / 2)])
    return paths


PlanetaryDesign.report = report_markdown            # type: ignore[attr-defined]
PlanetaryDesign.write_package = write_package       # type: ignore[attr-defined]
PlanetaryDesign.svg = layout_svg                    # type: ignore[attr-defined]

## Worked Example

**These are placeholder inputs**, chosen to show the method; replace them with the real envelope and shaft. A 120 mm diameter, 40 mm long envelope, a 14 mm keyed motor shaft, ratio 5, 20 N·m at the output, 3000 rpm in, 10 000 h:

In [ ]:
import time
req = PlanetaryRequirements(ratio=5.0, torque_out=20.0, n_in=3000.0, envelope_d=120.0, envelope_length=40.0,
                            shaft_d=14.0, life_h=10_000)
t0 = time.time()
designs = design_planetary(req)
print(f"{len(designs)} designs in {time.time() - t0:.1f} s")
for dd in designs:
    print(" ", dd.label())
best = designs[0]
assert best.rating.ok and best.components.ok and abs(best.ratio_error) <= req.ratio_tol

The chosen design, with all component checks:

In [ ]:
print(best.gears.summary())
print(best.rating.table())
for ch in best.components.checks:
    print(ch)

Its layout:

In [ ]:
from IPython.display import SVG, display
display(SVG(best.svg()))

And the manufacturing package written to a folder:

In [ ]:
import tempfile
with tempfile.TemporaryDirectory() as td:
    files = write_package(best, td)
    for f_ in files:
        print(os.path.basename(f_), os.path.getsize(f_), "bytes")
    print(open(os.path.join(td, "gear_data_planet.csv")).read()[:1500])

---

## Tests

In [ ]:
#| hide
# every design returned passes everything, and is sorted
for dd in designs:
    assert dd.rating.ok and dd.components.ok and dd.gears.ok
    assert dd.length <= req.envelope_length + 1e-9
keys = [(round(abs(x.ratio_error) / 0.005), round(x.length), -x.margin) for x in designs]
assert keys == sorted(keys)
# more torque never gives a shorter best design
req2 = replace(req, torque_out=60.0)
d2 = design_planetary(req2, max_results=1)
assert not d2 or d2[0].length >= best.length - 1e-9
# impossible envelope returns nothing rather than a failing design
assert design_planetary(replace(req, envelope_d=40.0)) == []
# the SVG is well formed XML
import xml.dom.minidom
xml.dom.minidom.parseString(best.svg())
# report mentions every check
rep = best.report()
for ch in best.components.checks:
    assert ch.name in rep